In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
from logging import INFO, StreamHandler, getLogger

logger = getLogger()
if not logger.hasHandlers():
    logger.addHandler(StreamHandler(sys.stdout))
logger.setLevel(INFO)

# Import


In [ ]:
import os
import sys
import pathlib
from itertools import product

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

ROOT_DIR = str(pathlib.Path(".").resolve().parent.parent)
print(f"ROOT_DIR: {ROOT_DIR}")
plt.rcParams["font.family"] = "serif"
plt.style.use("tableau-colorblind10")

In [ ]:
sys.path.append(ROOT_DIR)

# Define constants


In [ ]:
CESM_DATA_DIR = f"{ROOT_DIR}/data/cesm_le"
PROCESSED_DATA_DIR = f"{ROOT_DIR}/data/processed"

# Read data


In [ ]:
ds_20C_tref = xr.open_dataset(f"{CESM_DATA_DIR}/atm-monthly-cesmLE-20C-TREFHT.nc")
ds_RCP_tref = xr.open_dataset(f"{CESM_DATA_DIR}/atm-monthly-cesmLE-RCP85-TREFHT.nc")
ds_20C_prec = xr.open_dataset(f"{CESM_DATA_DIR}/atm-monthly-cesmLE-20C-PRECT.nc")
ds_RCP_prec = xr.open_dataset(f"{CESM_DATA_DIR}/atm-monthly-cesmLE-RCP85-PRECT.nc")

# Compare global distributions of metrics


In [ ]:
lon, lat = None, None
for ds in [ds_20C_tref, ds_RCP_tref, ds_20C_prec, ds_RCP_prec]:
    if lon is None:
        lon = ds.lon.values
        lat = ds.lat.values
    else:
        assert np.all(lon == ds.lon.values)
        assert np.all(lat == ds.lat.values)
lon, lat = np.meshgrid(ds.lon, ds.lat, indexing="ij")

In [ ]:
dict_metric_name = {
    "g_std": r"$\gamma_\mathrm{var}$",
    "g_sane": r"$\gamma_\mathrm{info}$",
    "g_sane_original": r"$\gamma_\mathrm{Sane,Org}$",
    "g_sane_ave_nbis": r"$\gamma_\mathrm{Sane}^{{\rm const_nbins}}$",
    "g_w": r"$\gamma_{W_1,{\rm pot}}$",
    "g_w_sort": r"$\gamma_{W_1}$",
}
# dict_value_scales = {
#     "TREFHT": {
#         "g_std": {"vmin": 0.13, "vmax": 0.84},
#         "g_sane": {"vmin": 0.48, "vmax": 0.94},
#         "g_w_sort": {"vmin": 0.02, "vmax": 0.45},
#     },
#     "PRECT": {
#         "g_std": {"vmin": 0.32, "vmax": 0.97},
#         "g_sane": {"vmin": 0.69, "vmax": 0.99},
#         "g_w_sort": {"vmin": 0.09, "vmax": 0.73},
#     },
# }
diff_cmap = "BuPu_r"

for (cmap, var_name), remove_annual_cycle in product(
    zip(
        ["viridis", "magma"],
        ["TREFHT", "PRECT"],
    ),
    ["remove_climatology"],
    # ["remove_climatology", "remove_annual_harmonic", "remove_running_mean", "none"],
):
    plt.rcParams["font.size"] = 13
    fig, axes = plt.subplots(
        3,
        3,
        figsize=(14, 7),
        sharex=True,
        sharey=True,
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    for j, metric_name in zip([0, 1, 2], ["g_std", "g_sane", "g_w_sort"]):
        dic = {}

        for i, kind in enumerate(["20C", "RCP85"]):
            out_name = f"atm-monthly-cesmLE-{kind}-{var_name}"
            if remove_annual_cycle == "remove_climatology":
                data = np.load(
                    f"{PROCESSED_DATA_DIR}/{out_name}_Ens00-40_no_annual_cycle_climatology.npz"
                )[metric_name]
            elif remove_annual_cycle == "remove_running_mean":
                data = np.load(
                    f"{PROCESSED_DATA_DIR}/{out_name}_Ens00-40_no_annual_cycle_running_mean.npz"
                )[metric_name]
            elif remove_annual_cycle == "remove_annual_harmonic":
                data = np.load(
                    f"{PROCESSED_DATA_DIR}/{out_name}_Ens00-40_no_annual_cycle_annual_harmonic.npz"
                )[metric_name]
            else:
                data = np.load(f"{PROCESSED_DATA_DIR}/{out_name}_Ens00-40.npz")[
                    metric_name
                ]

            dic[kind] = data

            ax = axes[i, j]

            ttl = "(20C)" if kind == "20C" else "(RCP8.5)"
            ax.set_title(f"{dict_metric_name[metric_name]} {ttl}")

            _vmin = np.quantile(data, 0.01)
            _vmax = np.quantile(data, 0.99)
            print(f"{var_name}, {kind}, {metric_name}: {_vmin=:.2f}, {_vmax=:.2f}")

            vmin = 0.0 if var_name == "TREFHT" else 0.5
            vmax = 1.0 if var_name == "TREFHT" else 1.0
            cnts = ax.pcolormesh(
                lon,
                lat,
                data,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                shading="nearest",
                transform=ccrs.PlateCarree(),
            )
            fig.colorbar(
                cnts,
                ax=ax,
                ticks=np.linspace(vmin, vmax, 5, endpoint=True),
                format="%.2f",
                extend=None if var_name == "TREFHT" else "min",
            )
            ax.set_global()
            ax.coastlines(lw=1.5)

        data = dic["RCP85"] - dic["20C"]
        _vmin = np.quantile(data, 0.01)
        _vmax = np.quantile(data, 0.99)
        print(f"{var_name}, diff {metric_name}: {_vmin=:.2f}, {_vmax=:.2f}")

        i = 2
        ax = axes[i, j]
        ttl = f"Diff. {dict_metric_name[metric_name]} " + r"(RCP8.5 $-$ 20C)"
        ax.set_title(ttl)

        vmax = 0.0
        vmin = -0.7 if var_name == "TREFHT" else -0.3

        cnts = ax.pcolormesh(
            lon,
            lat,
            data,
            cmap=diff_cmap,
            vmin=vmin,
            vmax=vmax,
            shading="nearest",
            transform=ccrs.PlateCarree(),
        )
        fig.colorbar(
            cnts,
            ax=ax,
            ticks=np.linspace(vmin, vmax, 5, endpoint=True),
            format="%.2f",
            extend="both",
        )
        ax.set_global()
        ax.coastlines(lw=1.5)
        print()

    if var_name == "TREFHT":
        ttl = "2 m Air Temperature"
    elif var_name == "PRECT":
        ttl = "Total Precipitation"
    else:
        raise ValueError(f"Invalid variable name: {var_name}")
    plt.suptitle(
        ttl
        + (
            ""
            if remove_annual_cycle == "remove_climatology"
            else f", {remove_annual_cycle}"
        )
    )
    plt.tight_layout()
    plt.show()

    if remove_annual_cycle == "remove_climatology":
        fig.savefig(
            f"{FIG_DIR}/cesm_metrics_all_members_{var_name}_no_annual_cycle.jpg",
            bbox_inches="tight",
        )

        os.makedirs("./figs", exist_ok=True)
        if var_name == "TREFHT":
            fig.savefig("./figs/fig04.pdf", bbox_inches="tight")
        elif var_name == "PRECT":
            fig.savefig("./figs/fig05.pdf", bbox_inches="tight")

In [ ]:
dict_metric_name = {
    "g_std": r"$\gamma_\mathrm{var}$",
    "g_sane": r"$\gamma_\mathrm{info}$",
    "g_sane_original": r"$\gamma_\mathrm{Sane,Org}$",
    "g_sane_ave_nbis": r"$\gamma_\mathrm{Sane}^{{\rm const_nbins}}$",
    "g_w": r"$\gamma_{W_1,{\rm pot}}$",
    "g_w_sort": r"$\gamma_{W_1}$",
    "g_w_sort_tot": r"$\widetilde{\gamma}_{W_1}$",
    "diff_ratio": r"$\widetilde{\gamma}_{W_1} - \gamma_{W_1}$",
}
# dict_value_scales = {
#     "TREFHT": {
#         "g_std": {"vmin": 0.13, "vmax": 0.84},
#         "g_sane": {"vmin": 0.48, "vmax": 0.94},
#         "g_w_sort": {"vmin": 0.02, "vmax": 0.45},
#     },
#     "PRECT": {
#         "g_std": {"vmin": 0.32, "vmax": 0.97},
#         "g_sane": {"vmin": 0.69, "vmax": 0.99},
#         "g_w_sort": {"vmin": 0.09, "vmax": 0.73},
#     },
# }

for (cmap, var_name), remove_annual_cycle in product(
    zip(
        ["viridis", "magma"],
        ["TREFHT", "PRECT"],
    ),
    ["remove_climatology"],
    # ["remove_climatology", "remove_annual_harmonic", "remove_running_mean", "none"],
):
    plt.rcParams["font.size"] = 13
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(14, 5),
        sharex=True,
        sharey=True,
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    for j, metric_name in zip([0, 1, 2], ["g_w_sort_tot", "g_w_sort", "diff_ratio"]):
        dic = {}

        for i, kind in enumerate(["20C", "RCP85"]):
            out_name = f"atm-monthly-cesmLE-{kind}-{var_name}"
            if remove_annual_cycle == "remove_climatology":
                if metric_name != "diff_ratio":
                    data = np.load(
                        f"{PROCESSED_DATA_DIR}/{out_name}_Ens00-40_no_annual_cycle_climatology.npz"
                    )[metric_name]
                    vmin = 0.0 if var_name == "TREFHT" else 0.5
                    vmax = 1.0 if var_name == "TREFHT" else 1.0
                else:
                    d1 = np.load(
                        f"{PROCESSED_DATA_DIR}/{out_name}_Ens00-40_no_annual_cycle_climatology.npz"
                    )["g_w_sort"]
                    d2 = np.load(
                        f"{PROCESSED_DATA_DIR}/{out_name}_Ens00-40_no_annual_cycle_climatology.npz"
                    )["g_w_sort_tot"]
                    data = d2 - d1
                    print(
                        np.nanquantile(data.flatten(), [0.01, 0.99]),
                        np.min(data),
                        np.max(data),
                    )
                    vmin = 0.0
                    vmax = 0.05
                    cmap = "YlOrRd"
            else:
                raise ValueError(f"Invalid remove_annual_cycle: {remove_annual_cycle}")

            dic[kind] = data

            ax = axes[i, j]

            ttl = "(20C)" if kind == "20C" else "(RCP8.5)"
            ax.set_title(f"{dict_metric_name[metric_name]} {ttl}")

            _vmin = np.quantile(data, 0.01)
            _vmax = np.quantile(data, 0.99)
            # print(f"{var_name}, {kind}, {metric_name}: {_vmin=:.2f}, {_vmax=:.2f}")

            cnts = ax.pcolormesh(
                lon,
                lat,
                data,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                shading="nearest",
                transform=ccrs.PlateCarree(),
            )
            fig.colorbar(
                cnts,
                ax=ax,
                ticks=(
                    np.linspace(vmin, vmax, 5, endpoint=True)
                    if metric_name != "diff_ratio"
                    else np.linspace(vmin, vmax, 6, endpoint=True)
                ),
                format="%.2f",
                extend=(
                    (None if var_name == "TREFHT" else "min")
                    if metric_name != "diff_ratio"
                    else None
                ),
            )
            ax.set_global()
            ax.coastlines(lw=1.5)

    if var_name == "TREFHT":
        ttl = "2 m Air Temperature"
    elif var_name == "PRECT":
        ttl = "Total Precipitation"
    else:
        raise ValueError(f"Invalid variable name: {var_name}")
    plt.suptitle(
        ttl
        + (
            ""
            if remove_annual_cycle == "remove_climatology"
            else f", {remove_annual_cycle}"
        )
    )
    plt.tight_layout()
    plt.show()

    if remove_annual_cycle == "remove_climatology":
        os.makedirs("./figs", exist_ok=True)
        if var_name == "TREFHT":
            fig.savefig("./figs/fig09.pdf", bbox_inches="tight")
            fig.savefig("./figs/fig09.jpg", bbox_inches="tight")
        elif var_name == "PRECT":
            fig.savefig("./figs/fig10.pdf", bbox_inches="tight")
            fig.savefig("./figs/fig10.jpg", bbox_inches="tight")

# Dependence on ensemble sizes


In [ ]:
lon, lat = None, None
for ds in [ds_20C_tref, ds_RCP_tref, ds_20C_prec, ds_RCP_prec]:
    if lon is None:
        lon = ds.lon.values
        lat = ds.lat.values
    else:
        assert np.all(lon == ds.lon.values)
        assert np.all(lat == ds.lat.values)
lon, lat = np.meshgrid(ds.lon, ds.lat, indexing="ij")

In [ ]:
dict_metric_name = {
    "g_std": r"$\gamma_\mathrm{var}$",
    "g_sane": r"$\gamma_\mathrm{info}$",
    "g_sane_original": r"$\gamma_\mathrm{Sane,Org}$",
    "g_sane_ave_nbis": r"$\gamma_\mathrm{Sane}^{\text{const nbins}}$",
    "g_w": r"$\gamma_{W_1,{\rm pot}}$",
    "g_w_sort": r"$\gamma_{W_1}$",
    "g_w_sort_tot": r"$\widetilde{\gamma}_{W_1}$",
}
# dict_value_scales = {
#     "TREFHT": {
#         "g_std": {"vmin": 0.13, "vmax": 0.84},
#         "g_sane": {"vmin": 0.48, "vmax": 0.94},
#         "g_w_sort": {"vmin": 0.02, "vmax": 0.45},
#     },
#     "PRECT": {
#         "g_std": {"vmin": 0.32, "vmax": 0.97},
#         "g_sane": {"vmin": 0.69, "vmax": 0.99},
#         "g_w_sort": {"vmin": 0.09, "vmax": 0.73},
#     },
# }

for kind, (cmap, var_name) in product(
    ["20C", "RCP85"], zip(["viridis", "magma"], ["TREFHT", "PRECT"])
):
    plt.rcParams["font.size"] = 13
    fig, axes = plt.subplots(
        3,
        4,
        figsize=(18, 7),
        sharex=True,
        sharey=True,
        subplot_kw={"projection": ccrs.PlateCarree()},
    )

    for j, metric_name in zip(
        [0, 1, 2, 3], ["g_std", "g_sane", "g_w_sort", "g_w_sort_tot"]
    ):
        for i, (ens_start, ens_end) in zip([0, 1, 2], [(0, 40), (0, 20), (20, 40)]):
            out_name = f"atm-monthly-cesmLE-{kind}-{var_name}"

            data = np.load(
                f"{PROCESSED_DATA_DIR}/{out_name}_Ens{ens_start:02}-{ens_end:02}_no_annual_cycle_climatology.npz"
            )[metric_name]

            ax = axes[i, j]
            ax.set_title(
                f"{dict_metric_name[metric_name]}  (Mem. = {ens_start+1:02}~{ens_end:02})"
            )

            vmin = 0.0 if var_name == "TREFHT" else 0.5
            vmax = 1.0 if var_name == "TREFHT" else 1.0

            cnts = ax.pcolormesh(
                lon,
                lat,
                data,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                shading="nearest",
                transform=ccrs.PlateCarree(),
            )
            fig.colorbar(
                cnts,
                ax=ax,
                ticks=np.linspace(vmin, vmax, 5, endpoint=True),
                format="%.2f",
                extend=None if var_name == "TREFHT" else "min",
            )
            ax.set_global()
            ax.coastlines(lw=1.5)

    if var_name == "TREFHT":
        ttl = "2 m Air Temperature " + ("(20C)" if kind == "20C" else "(RCP8.5)")
    elif var_name == "PRECT":
        ttl = "Precipitation " + ("(20C)" if kind == "20C" else "(RCP8.5)")
    else:
        raise ValueError(f"Invalid variable name: {var_name}")
    plt.suptitle(ttl)
    plt.tight_layout()
    plt.show()

    # if kind == "20C":
    #     fig.savefig(
    #         f"{FIG_DIR}/cesm_metrics_half_members_{kind}_{var_name}_no_annual_cycle.jpg",
    #         bbox_inches="tight",
    #     )